In [0]:

%sql
-- # KPI Aggregations
-- Revenue Performance by Region
CREATE OR REPLACE TEMP VIEW revenue_kpi AS
SELECT 
    region,
    SUM(total_sales) AS total_revenue,
    SUM(order_count) AS total_orders
FROM globalmart.gold.agg_monthly_revenue
GROUP BY region;

In [0]:
%sql
-- Vendor KPI ( aggregated and refine)

CREATE OR REPLACE TEMP VIEW vendor_kpi AS
SELECT 
    vendor_id,
    AVG(return_rate) AS avg_return_rate,
    SUM(total_returns) AS total_returns,
    SUM(total_orders) AS total_orders
FROM globalmart.gold.agg_vendor_performance
GROUP BY vendor_id;

In [0]:
%sql
-- Slow Moving KPI

CREATE OR REPLACE TEMP VIEW slow_kpi AS
SELECT 
    region,
    COUNT(product_id) AS slow_product_count,
    SUM(total_quantity_sold) AS total_quantity_sold
FROM globalmart.gold.agg_slow_moving_products
GROUP BY region;

In [0]:
%sql
SELECT * FROM revenue_kpi;


region,total_revenue,total_orders
Central,145492.72900000002,482
East,160101.2872,494


In [0]:
%sql
SELECT * FROM vendor_kpi;


vendor_id,avg_return_rate,total_returns,total_orders
VEN04,0.3023255813953488,13,43
VEN03,0.2958579881656805,50,169
VEN02,0.2771929824561403,158,570
VEN01,0.26288659793814434,51,194


In [0]:
%sql
SELECT * FROM slow_kpi;

region,slow_product_count,total_quantity_sold
Central,494,1998
East,470,1851


In [0]:
%sql
-- Demo 1
SELECT ai_query(
  'databricks-gpt-oss-20b',
  'Explain revenue trends in 3 lines'
);

"ai_query('databricks-gpt-oss-20b','Explain revenue trends in 3 lines')"
"Revenue has been steadily climbing over the past five years, driven by expanding market share and higher average selling prices. The growth rate, however, has begun to taper as competition intensifies and cost pressures rise. If current trends continue, revenue is expected to plateau and then shift toward diversification and new revenue streams."


In [0]:
%sql
-- Demo 2
SELECT ai_query(
  'databricks-gpt-oss-20b',
  CONCAT(
    'Analyze vendor return rates: ',
    to_json(collect_list(struct(vendor_id, avg_return_rate)))
  )
)
FROM vendor_kpi;

"ai_query('databricks-gpt-oss-20b',CONCAT('Analyze vendor return rates: ',to_json(collect_list(STRUCT(vendor_id,avg_return_rate)))))"
"**Vendor Return‑Rate Snapshot (average, 2025‑Q4)** | Rank | Vendor | Avg Return Rate | Relative to Top Vendor | |------|--------|-----------------|------------------------| | 1 | **VEN04** | **30.23 %** | 1.00× | | 2 | **VEN03** | 29.59 % | 0.98× | | 3 | **VEN02** | 27.72 % | 0.92× | | 4 | **VEN01** | 26.29 % | 0.87× | --- ### Key Takeaways | Insight | What it means | Suggested Action | |---------|----------------|------------------| | **High return rates across the board** | All vendors exceed the industry benchmark of ~15‑20 % for consumer goods. | Investigate root causes (product quality, description accuracy, shipping damage). | | **VEN04 is the highest** | 30 % return rate is 4 % higher than the next best vendor. | Target a 10‑15 % reduction in the next 6 months via stricter quality checks and clearer product specs. | | **Small gap between VEN03 and VEN04** | The difference is only 0.64 %. | A quick win: share best‑practice SOPs from VEN04 to VEN03 (e.g., packaging, inspection). | | **VEN01 has the lowest** | 26.3 % still above target but comparatively better. | Use VEN01’s processes as a benchmark; consider a “quality champion” program. | | **Overall trend** | Return rates are trending upward by ~0.5 % per quarter (based on historical data). | Implement a quarterly return‑rate review cycle and tie vendor incentives to improvement. | --- ### Action Plan (next 90 days) 1. **Root‑Cause Analysis (RCA) – 2 weeks** * Pull return‑reason codes for each vendor. * Identify top 3 reasons per vendor (e.g., “defective”, “wrong size”, “poor packaging”). 2. **Vendor‑Specific Improvement Plans** * **VEN04** – Focus on packaging and inspection. * **VEN03** – Align product descriptions with actual items. * **VEN02** – Strengthen quality control at the source. * **VEN01** – Maintain current practices; share with others. 3. **Performance Metrics** * Set a 10 % reduction target for each vendor. * Track monthly return‑rate vs. target; flag >5 % variance. 4. **Incentives & Penalties** * Offer a 2 % margin bonus for vendors who hit the target. * Apply a 1 % margin penalty for vendors who exceed the target by >5 %. 5. **Continuous Improvement** * Quarterly “Return‑Rate Review” meetings. * Share best practices and lessons learned across vendors. --- ### Quick Wins | Quick Win | Why it works | How to implement | |-----------|--------------|------------------| | **Standardized Return Labels** | Reduces confusion, speeds up processing. | Issue a single, branded return label to all vendors. | | **Pre‑Shipment Inspection Checklist** | Catches defects before shipping. | Provide a digital checklist; require sign‑off before dispatch. | | **Customer Feedback Loop** | Direct insight into why returns happen. | Add a short survey on the return confirmation page. | --- **Bottom line:** While all vendors are above the industry return‑rate target, the differences are modest. A focused, data‑driven improvement program can bring all rates below 25 % within a year, boosting customer satisfaction and margin."


In [0]:
%sql
CREATE TABLE IF NOT EXISTS globalmart.gold.ai_business_insights (
    insight_type STRING,
    summary STRING,
    kpi_data STRING,
    generated_at TIMESTAMP
);

In [0]:
%sql
DELETE FROM globalmart.gold.ai_business_insights;

num_affected_rows
3


In [0]:
%sql
INSERT INTO globalmart.gold.ai_business_insights
SELECT
  'revenue_performance',
  ai_query(
    'databricks-gpt-oss-20b',
    CONCAT(
      'You are a senior retail executive. Provide a 4-6 sentence executive summary highlighting key revenue trends, top-performing regions, and risks: ',
      to_json(collect_list(struct(region, total_revenue, total_orders)))
    )
  ),
  to_json(collect_list(struct(region, total_revenue, total_orders))),
  current_timestamp()
FROM revenue_kpi;

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO globalmart.gold.ai_business_insights
SELECT
  'vendor_return_rate',
  ai_query(
    'databricks-gpt-oss-20b',
    CONCAT(
      'Analyze vendor return rates. Identify high-return vendors, possible quality issues, and business risks. Provide a 4-6 sentence executive summary: ',
      to_json(collect_list(struct(vendor_id, avg_return_rate, total_returns, total_orders)))
    )
  ),
  to_json(collect_list(struct(vendor_id, avg_return_rate, total_returns, total_orders))),
  current_timestamp()
FROM vendor_kpi;

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO globalmart.gold.ai_business_insights
SELECT
  'slow_moving_inventory',
  ai_query(
    'databricks-gpt-oss-20b',
    CONCAT(
      'Analyze slow-moving inventory across regions. Highlight overstock risks and underperforming regions. Provide a 4-6 sentence executive summary: ',
      to_json(collect_list(struct(region, slow_product_count, total_quantity_sold)))
    )
  ),
  to_json(collect_list(struct(region, slow_product_count, total_quantity_sold))),
  current_timestamp()
FROM slow_kpi;

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT * FROM globalmart.gold.ai_business_insights;

insight_type,summary,kpi_data,generated_at
vendor_return_rate,"Executive Summary: The return analysis reveals that three vendors—VEN04, VEN02, and VEN03—exhibit return rates above 27 %, with VEN04 leading at 30.23 % (13 returns from 43 orders). VEN02 follows closely at 27.72 % (158 returns from 570 orders), while VEN03’s 29.59 % return rate (50 returns from 169 orders) also signals potential quality or fulfillment issues. In contrast, VEN01’s return rate of 26.29 % (51 returns from 194 orders) is comparatively lower but still above the industry benchmark. These elevated return rates suggest recurring product defects, packaging failures, or misaligned product specifications, posing risks to revenue, inventory turnover, and customer satisfaction. Immediate investigation into the root causes—such as supplier quality controls, inspection processes, and post‑sale support—will be essential to mitigate losses and protect brand reputation.","[{""vendor_id"":""VEN04"",""avg_return_rate"":0.3023255813953488,""total_returns"":13,""total_orders"":43},{""vendor_id"":""VEN02"",""avg_return_rate"":0.2771929824561403,""total_returns"":158,""total_orders"":570},{""vendor_id"":""VEN01"",""avg_return_rate"":0.26288659793814434,""total_returns"":51,""total_orders"":194},{""vendor_id"":""VEN03"",""avg_return_rate"":0.2958579881656805,""total_returns"":50,""total_orders"":169}]",2026-03-26T15:29:59.797Z
slow_moving_inventory,"**Executive Summary** Across the two key markets, the East and Central regions exhibit significant slow‑moving inventory, with 470 and 494 slow‑product categories respectively. The Central region’s higher count of slow items, coupled with a modestly higher average sale of 4.0 units per slow product, signals a greater overstock risk than the East, which averages only 3.9 units per slow product. In absolute terms, the Central region has sold 1,998 units from its slow‑moving stock, slightly more than the East’s 1,851 units, yet the larger inventory base dilutes overall efficiency. The East region’s lower sales velocity per slow product indicates it is the more underperforming of the two, suggesting a need for targeted promotion or markdown strategies. Both regions should prioritize inventory optimization, with Central focusing on reducing excess stock and East concentrating on boosting demand for its slow‑moving categories.","[{""region"":""East"",""slow_product_count"":470,""total_quantity_sold"":1851},{""region"":""Central"",""slow_product_count"":494,""total_quantity_sold"":1998}]",2026-03-26T15:30:13.996Z
revenue_performance,"Executive Summary The East region remains the top‑performing market, generating $160,101 in revenue from 494 orders, a 10.5% lift over the Central region’s $145,493 from 482 orders. The modest 2.6% higher order volume in the East underpins its revenue advantage, suggesting stronger customer engagement and higher average order value. Revenue growth across both regions is steady, but the narrow margin between them signals a potential plateau in market penetration. Key risks include over‑reliance on these two geographic segments, exposure to regional supply‑chain disruptions, and the possibility of diminishing returns as competition intensifies. Diversifying into under‑served regions and reinforcing supply‑chain resilience will be critical to sustaining growth momentum.","[{""region"":""East"",""total_revenue"":160101.2872,""total_orders"":494},{""region"":""Central"",""total_revenue"":145492.72900000002,""total_orders"":482}]",2026-03-26T15:29:37.809Z
